# Time Series Forecasting with snowflakeR

An end-to-end forecasting workflow: query Snowflake data, train an ARIMA model
in R, register it to the Model Registry, and run inference via SPCS.

**What you'll do:**
1. Query TPC-H sample data from Snowflake
2. Train a time series model with `forecast::auto.arima()`
3. Visualize decomposition and forecasts with ggplot2
4. Register the model with `sfr_log_model()` (custom predict logic)
5. Deploy and run inference via SPCS

**Key concept:** For models that don't follow the standard `predict(model, newdata)`
interface, `snowflakeR` supports a `predict_body` template. The `forecast` package
uses `forecast(model, h = N)` instead, so we supply custom R code that runs both
locally (via `sfr_predict_local()`) and remotely (in the SPCS container).

This notebook is for **Snowflake Workspace Notebooks** (Python kernel + `%%R` magic).
For local environments (RStudio, Posit, JupyterLab), use `local_forecasting_demo.ipynb`.

**Before you start:** Copy `notebook_config.yaml.template` to `notebook_config.yaml`
and edit it with your warehouse, database, and schema.

**Sections:**
1. [Setup](#section-1)
2. [Connect](#section-2)
3. [Data Exploration](#section-3)
4. [Time Series Preparation](#section-4)
5. [Model Training](#section-5)
6. [Test Locally](#section-6)
7. [Register to Snowflake](#section-7)
8. [Deploy & Inference](#section-8)
9. [Visualize Forecasts](#section-9)
10. [Cleanup](#section-10)

## 1. Setup

### Step 1: Install R environment (~3 minutes, first time only)

In [ ]:
from sfnb_setup import install_r
install_r(config="snowflaker_forecast_config.yaml")

### Step 2: Install snowflakeR

In [ ]:
from sfnb_setup import install_r_packages
install_r_packages(config="snowflaker_forecast_config.yaml", packages=["snowflakeR"])

In [ ]:
%%R
library(forecast)
library(ggplot2)
library(dplyr)
cat("forecast", as.character(packageVersion("forecast")), "loaded\n")

---

## 2. Connect & Set Execution Context

In [ ]:
%%R
conn <- sfr_connect()
conn <- sfr_load_notebook_config(conn)
conn

reg <- sfr_model_registry(conn)
reg

---

## 3. Data Exploration

Query monthly order volume from the TPC-H sample dataset (available in every
Snowflake account at `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`).

In [ ]:
%%R
orders_df <- sfr_query(conn, "SELECT
  DATE_TRUNC('MONTH', O_ORDERDATE) AS ORDER_MONTH,
  COUNT(*)                          AS ORDER_COUNT,
  SUM(O_TOTALPRICE)                 AS TOTAL_REVENUE,
  AVG(O_TOTALPRICE)                 AS AVG_ORDER_VALUE
  FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
  GROUP BY 1
  ORDER BY 1
")

orders_df$ORDER_MONTH <- as.Date(orders_df$ORDER_MONTH)
rcat(sprintf("%d months of data: %s to %s",
             nrow(orders_df),
             min(orders_df$ORDER_MONTH),
             max(orders_df$ORDER_MONTH)))
head(orders_df, 10)

In [ ]:
%%R -w 900 -h 400
library(scales)

ggplot(orders_df, aes(x = ORDER_MONTH, y = ORDER_COUNT)) +
  geom_line(color = "steelblue", linewidth = 1) +
  geom_point(color = "steelblue", size = 2) +
  scale_y_continuous(labels = comma) +
  scale_x_date(date_breaks = "1 year", date_labels = "%Y") +
  labs(title = "Monthly Order Volume (TPC-H)",
       subtitle = "Time series data for ARIMA forecasting",
       x = "Month", y = "Order Count") +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"))

---

## 4. Time Series Preparation

Convert to an R `ts` object and decompose into trend, seasonal, and remainder.

In [ ]:
%%R
orders_df <- orders_df[order(orders_df$ORDER_MONTH), ]

start_date  <- min(orders_df$ORDER_MONTH)
start_year  <- as.numeric(format(start_date, "%Y"))
start_month <- as.numeric(format(start_date, "%m"))

orders_ts <- ts(orders_df$ORDER_COUNT,
                start = c(start_year, start_month),
                frequency = 12)

cat(sprintf("Time series: %d observations, %s %d to %s %d\n",
            length(orders_ts),
            month.abb[start(orders_ts)[2]], start(orders_ts)[1],
            month.abb[end(orders_ts)[2]], end(orders_ts)[1]))

In [ ]:
%%R -w 900 -h 500
decomp <- stl(orders_ts, s.window = "periodic")
plot(decomp, main = "STL Decomposition: Monthly Order Volume")

---

## 5. Model Training

`auto.arima()` performs an automatic search over ARIMA(p,d,q)(P,D,Q) parameter
space and selects the best model by AICc. We hold out the last 12 months to
evaluate accuracy before training the final model on all data.

In [ ]:
%%R
n_test  <- 12
n_train <- length(orders_ts) - n_test

train_ts <- head(orders_ts, n_train)
test_ts  <- tail(orders_ts, n_test)

cat(sprintf("Training: %d months | Test: %d months\n", n_train, n_test))

In [ ]:
%%R
cat("Training ARIMA model (stepwise = FALSE for thorough search)...\n")
arima_model <- auto.arima(train_ts, seasonal = TRUE, stepwise = FALSE)
summary(arima_model)

In [ ]:
%%R -w 900 -h 400
fc <- forecast(arima_model, h = n_test)

autoplot(fc) +
  autolayer(test_ts, series = "Actual", color = "red") +
  labs(title = "ARIMA Forecast vs Actual (holdout)",
       subtitle = arima_model$method,
       x = "Time", y = "Order Count") +
  theme_minimal() +
  theme(legend.position = "bottom")

In [ ]:
%%R
cat("Holdout accuracy:\n")
print(accuracy(fc, test_ts))

### Train final model on all data

In [ ]:
%%R
cat("Training final model on full dataset...\n")
final_model <- auto.arima(orders_ts, seasonal = TRUE, stepwise = FALSE)
print(final_model)

---

## 6. Test Locally

The `forecast` package uses `forecast(model, h = N)` instead of the standard
`predict(model, newdata)` interface. We supply a `predict_body` template
that `sfr_predict_local()` executes locally and `sfr_log_model()` bakes into
the SPCS container.

**Convention:** pass N rows to get an N-period forecast. The input data frame
contains a `period` column (1..N); the actual values are unused -- only the
row count matters.

In [ ]:
%%R
forecast_body <- '
  pred_{{UID}} <- forecast::forecast({{MODEL}}, h = {{N}})
  result_{{UID}} <- data.frame(
    period         = seq_len({{N}}),
    point_forecast = as.numeric(pred_{{UID}}$mean),
    lower_80       = as.numeric(pred_{{UID}}$lower[, 1]),
    upper_80       = as.numeric(pred_{{UID}}$upper[, 1]),
    lower_95       = as.numeric(pred_{{UID}}$lower[, 2]),
    upper_95       = as.numeric(pred_{{UID}}$upper[, 2])
  )
'

test_input <- data.frame(period = 1:6)

local_preds <- sfr_predict_local(
  final_model, test_input,
  predict_pkgs = c("forecast"),
  predict_body = forecast_body
)

rcat("Local 6-period forecast:")
local_preds

---

## 7. Register to Snowflake

`sfr_log_model()` handles everything: saves the R model to `.rds`, auto-generates
a Python `CustomModel` wrapper that uses the `predict_body` template, and registers
it in the Snowflake Model Registry.

The `conda_deps` include `r-forecast` so the SPCS container has the forecast
package pre-installed.

In [ ]:
%%R
mv <- sfr_log_model(
  reg,
  model        = final_model,
  model_name   = "SFR_TPCH_FORECAST",
  predict_pkgs = c("forecast"),
  predict_body = forecast_body,
  input_cols   = list(period = "integer"),
  output_cols  = list(
    period         = "integer",
    point_forecast = "double",
    lower_80       = "double",
    upper_80       = "double",
    lower_95       = "double",
    upper_95       = "double"
  ),
  conda_deps = c("r-forecast>=8.0", "numpy<2.0"),
  comment    = "ARIMA forecast model for TPC-H monthly orders (auto.arima)"
)

mv

In [ ]:
%%R
models <- sfr_show_models(reg)
models

### Attach accuracy metrics

In [ ]:
%%R
acc <- accuracy(fc, test_ts)
sfr_set_model_metric(reg, "SFR_TPCH_FORECAST", mv$version_name,
                     "test_rmse", acc["Test set", "RMSE"])
sfr_set_model_metric(reg, "SFR_TPCH_FORECAST", mv$version_name,
                     "test_mape", acc["Test set", "MAPE"])

rcat(sprintf("Metrics set -- RMSE: %.1f, MAPE: %.2f%%",
             acc["Test set", "RMSE"], acc["Test set", "MAPE"]))

---

## 8. Deploy & Inference via SPCS

Create SPCS infrastructure (if needed), deploy the model as a service,
and run a 12-month forecast.

In [ ]:
%%R
sfr_create_compute_pool(conn, "R_FORECAST_POOL", instance_family = "CPU_X64_M")
sfr_create_image_repo(conn, sfr_fqn(conn, "R_FORECAST_IMAGES"))

sfr_deploy_model(
  reg,
  model_name   = "SFR_TPCH_FORECAST",
  version_name = mv$version_name,
  service_name = "tpch_forecast_svc",
  compute_pool = "R_FORECAST_POOL",
  image_repo   = sfr_fqn(conn, "R_FORECAST_IMAGES"),
  force        = TRUE
)

In [ ]:
%%R
sfr_wait_for_service(reg, "tpch_forecast_svc", timeout_min = 10, poll_sec = 15)

In [ ]:
%%R
inference_input <- data.frame(period = 1:12)

preds <- sfr_predict(
  reg, "SFR_TPCH_FORECAST", inference_input,
  service_name = "tpch_forecast_svc"
)

rcat("12-month SPCS forecast:")
preds

---

## 9. Visualize Forecasts

Plot historical data alongside the 12-month forecast with 80% and 95%
confidence intervals.

In [ ]:
%%R -w 900 -h 500
library(scales)

last_date <- max(orders_df$ORDER_MONTH)
preds$forecast_date <- seq.Date(from = last_date + 30,
                                by = "month",
                                length.out = nrow(preds))

ggplot() +
  geom_ribbon(data = preds,
              aes(x = forecast_date, ymin = lower_95, ymax = upper_95),
              fill = "steelblue", alpha = 0.2) +
  geom_ribbon(data = preds,
              aes(x = forecast_date, ymin = lower_80, ymax = upper_80),
              fill = "steelblue", alpha = 0.3) +
  geom_line(data = orders_df,
            aes(x = ORDER_MONTH, y = ORDER_COUNT),
            color = "black", linewidth = 1) +
  geom_line(data = preds,
            aes(x = forecast_date, y = point_forecast),
            color = "steelblue", linewidth = 1, linetype = "dashed") +
  geom_point(data = preds,
             aes(x = forecast_date, y = point_forecast),
             color = "steelblue", size = 2) +
  scale_y_continuous(labels = comma) +
  scale_x_date(date_breaks = "1 year", date_labels = "%Y") +
  labs(title = "TPC-H Orders: 12-Month Forecast",
       subtitle = "ARIMA model via snowflakeR + SPCS (80% and 95% CI)",
       x = "Date", y = "Order Count") +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"))

---

## 10. Cleanup

Uncomment the lines below to remove demo resources.

In [ ]:
%%R
# sfr_undeploy_model(reg, "SFR_TPCH_FORECAST", mv$version_name, "tpch_forecast_svc")
# sfr_delete_model(reg, "SFR_TPCH_FORECAST")
# sfr_execute(conn, "DROP COMPUTE POOL IF EXISTS R_FORECAST_POOL")
# sfr_execute(conn, sfr_fqn(conn, "DROP IMAGE REPOSITORY IF EXISTS R_FORECAST_IMAGES"))
cat("Cleanup section -- uncomment lines above to delete resources\n")

---

## Summary

This notebook demonstrated:

1. **Querying Snowflake data** with `sfr_query()` for time series analysis
2. **Training an ARIMA model** with R's `forecast::auto.arima()`
3. **Custom predict logic** via the `predict_body` template for non-standard models
4. **Local testing** with `sfr_predict_local()` before registering
5. **One-call registration** with `sfr_log_model()` (no Python code needed)
6. **SPCS deployment** with `sfr_deploy_model()` and `sfr_wait_for_service()`
7. **Forecast visualization** with ggplot2 confidence intervals

### Key Takeaway

The `predict_body` template bridges R models that don't follow the standard
`predict(model, newdata)` convention. The same template code runs identically
in `sfr_predict_local()` (for testing) and in the SPCS container (for production).

### Next Steps

- Try ETS (`ets()`), Prophet, or other forecasting models
- Add exogenous variables with ARIMAX
- Schedule recurring forecasts with Snowflake Tasks
- See `workspace_model_registry.ipynb` for the full Model Registry feature tour